# Miary jakości modelu

W tym ćwiczeniu skorzystamy z modułu sklearn [scikit-learn](https://scikit-learn.org/stable/auto_examples/index.html) (importujemy go jako `sklearn`) przeznaczonego do uczenia maszynowego, w szczególności do modeli klasycznych. Zawiera wiele zoptymalizowanych i przydatnych funkcji i algorytmów. Alternatywnymi frameworkami, nastawionymi głównie na sieci neuronowe, są np. [PyTorch](https://pytorch.org/tutorials/), [Tensorflow](https://www.tensorflow.org/tutorials), [Keras](https://keras.io/examples/), [Caffe2](https://caffe2.ai/docs/tutorials), chociaż z obserwacji środowiska wynika, że dominują raczej PyTorch i Tensorflow.


Na stronie przedmiotu znajduje się instrukcja konfiguracji środowiska w przypadku pracy lokalnej, a nie w colabie. Może (nie musi) być przydatna dla osób pracujących na własnych laptopach:

https://brain.fuw.edu.pl/edu/index.php/Uczenie_maszynowe_i_sztuczne_sieci_neuronowe/konfiguracja

### Przygotowanie środowiska programistycznego

In [ ]:
import sklearn
print('Zainstalowana wersja scikit-learn: {}.'.format(sklearn.__version__))

import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import pandas as pd

### Przygotowanie syntetycznych danych

Nasze przykłady będą należały do dwu klas: **0** lub **1**. Wartości zmiennych wejściowych $(x, y)$ dla danej klasy są dane dwuwymiarowym rozkładem Gaussa:

$$
p_{class~1}(x,y) = N(\mu_{class~1, x}, \mu_{class~1, y}, \text{covariance})
$$

W bibliotece numpy wielowymiarowe rozkłady Gaussa mona uzyskać za pomocą funkcji `np.random.default_rng().multivariate_normal()`

**Proszę:** 

* uzupełnić kod funkcji ```generate_class_points(nPoints, means, covariance, class)``` która zwraca macierz $(nPoints, 3)$ zawierającą położenia punktów danej klasy, oraz jej numer: (x,y,class) funkcja na wejściu przyjmuje:
    * liczbę przykładów do generacji $nPoints$, macierze $means$ i $covariance$ zawierające średnie i kowariancję dla danej klasy:
$$
means = [\mu_{x}, \mu_{y}]
$$
$$
covariance = 
\begin{bmatrix}
    \sigma^{2}_{x} & \sigma_{xy} \\
    \sigma_{xy} & \sigma^{2}_{y}
\end{bmatrix}   
$$
    * liczbę kodującą numer klasy : $class$, w naszym przypadku 0 lub 1
 
 
**Proszę:**

używając z funkcji  ```generate_class_points(nPoints, means, covariance)``` wygenenerować po 5000 punktów dla dwu klas, dla których parametry rozkładów Gaussa są następujące:

```Python
means_class_0 = np.array([1,5]) 
covariance_class_0 = np.diag([3,3]) 

means_class_1 = np.array([4,7]) 
covariance_class_1 = np.diag([3,2]) 
```

Dane proszę załadować do obiektów pandas DataFrame dla każdej z klas, a następnie je połączyć korzystając z funkcji ```pandas.concatenate()```.

In [ ]:
def generate_class_data(nPoints, means, covariances, label):
    #Generacja własności - w naszym wypadku wartości (x,y) dla danej klasy
...
    #Generacja macierzy z etykietami - tutaj numerem klasy
    labels = np.full((nPoints,1), label)
    #Połączenie macierzy dla własności i etykiet w jedną macierz
    class_data = np.concatenate((features, labels), axis=1)
    return class_data
    
nPoints = 5000
    
#Defincja parametrów rozkładu Gaussa dla klasy "0"    
...

#Definicja parametrów rozkładu Gaussa dla klasy "1"
...

#Generacja danych dla klas "0" i "1"
...

#Sprawdzenie czy wygenerowane dane mają poprawny kształt
print("class_0_data.shape ",class_0_data.shape)
print("class_1_data.shape ",class_1_data.shape)

#Utworzenie obiektu DataFrame dla klas "0" i "1"
...

#Utworzenie obiektu DataFrame dla obu klas łącznie
df = pd.concat([df_class_0, df_class_1], ignore_index=True)
#Wymieszanie obiektu DataFrame dla obu klas łącznie
df = df.sample(frac=1, ignore_index=True)
#Wypisanie zawartości obiektu DataFrame dla obu klas łącznie
print(df)

## Analiza wizualna danych. 

Pierwszy krok przy analizie danych z użyciem dowolnego algorytmu to ich inspekcja. Korzystając z funkcji biblioteki `plotly.express`:
* narysować rozkłady wszystkich zmiennych wejściowych oddzielnie dla klas 0 i 1 (`px.histogram()` - Zwróć uwagę na parametr barmode, zobacz co się stanie dla "group" a co jak go zmienisz na "overlay".)
* narysować wykres korelacji między zmiennymi wejściowymi dla klas 0 i 1 na jednym rysunku (`px.scatter()`)

In [ ]:

...


# Przypadek klas równolicznych

* podziel pełne dane na podzbiory uczący (80% danych) i treningowy (20% danych). Wykonując podział zwróć uwagę na randomizację danych.
* narysuj dwuwymiarowe rozkłady dla zbiorów uczącego i treningowego i sprawdź wizualnie czy są różne. 

**Wskazówka:** do podziałów zbioru proszę użyć funkcji [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) podając obiekt DataFrame jako argument.

In [ ]:

...


## Trening regresji logistycznej

Skorzystamy z implementacji regresji logistycznej z pakietu scikit-learn.
Regresja logistyczna zaimplementowana jest w klasie [`LogisticRegression`](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). 

**Proszę:**

* stworzyć obiekt klasy ```LogisticRegression()```
* użyć tego obiektu by wytrenować model. W tym miejscu wykonają się kroki odpowiadające procedurze minimalizacji znajdującej parametry $\theta$ z zajęć o regresji logistycznej
* przeprowadzić predykcję na zbiorze testowym

**Wskazówka:** domyślną funkcją minimalizacji jest 'lbfgs', ale warto ją podać explicite by uniknąć uciążliwego ostrzeżenia.

In [ ]:
#Definicja modelu - obiektu klasy LogisticRegression
...

#Przeprowadzenie minmalizacji, inaczej dopasowania, parametrów modelu (ang. fit) trenowania modelu
...

#Przeprowadzenie predykcji. Czym różnią się te dwie opcje? 
y_pred = model.predict(df_test[["x","y"]])
y_prob = model.predict_proba(df_test[["x","y"]])
print("y_pred:", y_pred)
print("y_prob:\n", y_prob)

**Proszę**

* obliczyć i narysować macierz pomyłek (ang. confusion matrix)

**Wsakzówki:**
* skorzystaj z funkcji ```metrics.confusion_matrix()``` by wypisać wartości macierzy
* skorzystaj z funkcji ```ConfusionMatrixDisplay.from_estimator()``` by narysować macierz pomyłek podając model jako argument funkcji. 
  * Narysuj macierz zwykłą oraz znormalizowaną (znajdź odpowiedni argument w dokumentacji). 
  * Zwróć uwagę na różne wersje normalizacji. Porównaj z [tabelą ](https://en.wikipedia.org/wiki/Receiver_operating_characteristic#Basic_concept).

**Pytanie**: jaką iterpretację mają elementy macierzy po normalizacji?

In [ ]:
#Obliczenie macierzy pomyłek
cm = confusion_matrix(df_test["label"], y_pred)
#Wypisanie macierzy pomyłek w postaci tekstowej
...

#Narysowanie macierzy pomyłek. 
...

#Narysowanie macierzy pomyłek z normalizacją względem liczny przykładów w prawdziwych przykładów.
...
##

Zdefiniujmy następujące przypadki gdy nasz model się myli lub podaje poprawny wynik:

* **"True Positive" (TP)**:  stan faktyczny jest pozytywny (y=1) i klasyfikator się nie myli (wynik = 1)
* **"True Negative" (TN)**:  stan faktyczny jest negatywny (y=0) i klasyfikator się nie myli (wynik = 0) 
* **"False Positive" (FP)**: wynik fałszywie pozytywny (fałszywy alarm): stan faktyczny jest negatywny (y=0) ale klasyfikator się  myli (wynik = 1)
* **"False Netative" (FN)**: przegapiony alarm: stan faktyczny jest pozytywny (y=1) i klasyfikator się myli (wynik = 0)

Zbadajmy wartości różnych miar jakości:

* **precyzja pozytywna - _precision_ (positive predictive value (PPV)):**

$\qquad$ $PPV = \frac{TP}{P'}=\frac{TP}{ TP + FP}$

Ułamek przypadków oznaczonych jako klasa "1" który naprawdę należy do klasy "1". Ułamek ten jest np. odpowiedzą na pytanie
"Jeśli wynik testu jest pozytywny, jakie jest prawdopodobieństwo, że osoba badana jest chora?"

* **czułość - _recall_ (ang. True Positive Rate, TPR):** 

$\qquad$ $TPR = \frac{TP}{ P} = \frac{TP} { TP+FN}$

Ułamek przypadków które należą do klasy "1" i został poprawnie oznaczony. Ułamek ten jest np. odpowiedzą na pytanie  "Jakie jest prawdopodobieństwo, że test wykonany dla osoby chorej wykaże, że jest ona chora?"


* **dokładność - _accuracy_ (ACC)):** Prawdopodobieństwo prawidłowej klasyfikacji.

$\qquad$ $ACC = \frac{TP + TN}{P + N}$

Ułamek przypadków, niezależnie od klasy, które zostały poprawnie oznaczone. Ułamek ten jest np. odpowiedzą na pytanie  "Jakie jest prawdopodobieństwo, że test poda poprawną odpowiedź?" 

* **F1-score:** średnia harmoniczna z precyzji i czułości:

$\qquad$ $F_1= 2 \frac{PPV  \cdot TPR}{PPV+TPR}= \frac{2TP}{ 2TP+FP+FN}$

Miara ta daje ocenę balansu między czułością a precyzją. Miara ta nie uwzględnia wyników prawdziwie negatywnych.

* **współczynnik korelacji Matthewsa ( Matthews correlation coefficient):**

$\qquad$ $
\text{MCC} = \frac{ TP \cdot TN - FP \cdot FN } {\sqrt{ (TP + FP) ( TP + FN ) ( TN + FP ) ( TN + FN ) } }
$

Współczynnik MMC uwzględnia wyniki zarówno prawdziwie jaki i fałszywie pozytywne i negatywne i jest na ogół uważany jako zrównoważona miara, która może być stosowana nawet wtedy, gdy klasy są bardzo różnej liczebności. 

MCC jest w istocie współczynnikiem korelacji pomiędzy obserwowanymi i przewidywanymi klasyfikacjami binarnymi; zwraca wartość od -1 do +1. 

* Współczynnik +1 odpowiada idealnej klasyfikacji, 
* 0 nie lepiej niż losowe przypisanie wyniku 
* -1 oznacza całkowitą niezgodę między klasyfikacją  i stanem faktycznym.
 
***
 
**Proszę** 

uzupełnić funkcje:
*  ```train_model(X_test, X_train, y_train)```, która trenuje model ```LogisticRegression``` i zwraca **prawdopodobieństwa** wyestymowane dla X_test
* ```get_scores(y_true, y_prob, th=0.5)```, która zwraca słownik z miarami jakości modelu dla otrzymanych prawdopodobieństw. Parametr `th` to próg prawdopodobieństwa klasy 1 powyżej którego przypiszemy kalse 1 danemu przykładowi. 

**Wskazówka** Opisane powyżej miary są dostępne w bibliotece [sklearn](https://scikit-learn.org/stable/api/sklearn.metrics.html#module-sklearn.metrics)

In [ ]:
%%time
def train_model(X_test, X_train, y_train):
    # Inicjalizacja klasy modelu
...
    # Dopasowanie modelu do danych treningowych
...

    #Predykcja na zbiorze testowym
...

    return y_prob

def get_scores(y_true, y_prob, th=0.5):
    #Przekształcenie prawdopodobieństw na etykiety klas
...
    #Obliczenie wartości miar jakości
    metrics_dict = {}
...
    return metrics_dict
  

y_prob = train_model(df_test[["x","y"]], df_train[["x","y"]], df_train["label"])  

metrics_dict = get_scores(df_test["label"], y_prob)
print(metrics_dict)


W czasie analizy wydajności modelu zwykle stosuje się modyfikację losowego podziału na dane treningowe i testowe. Całe dane są dzielone na
równych ```k``` podziałów (ang. folds) i jest przeprowadzana k-krotna procedura sprawdzania (ang. k-fold cross-validation, pl. walidacja krzyżowa):

0. dzielimy zbiór uczący (features, labels) na `k` równych części
1. odkładamy i-tą, i=(0,1,2,3,...,k-1), część jako dane testowe, 
2. na pozostałych `k-1` częściach uczymy klasyfikator
3. obliczamy miary jakości na tej odłożonej części
4. wracamy do punktu 1
5. na końcu mozna policzyć wartości metryk uśrednione po podziałach

_**Co to jest stratyfikacja?**
Stratyfikacja to metoda podziału danych, która **zachowuje proporcje klas** w zbiorach treningowym i testowym takie same jak w oryginalnym zbiorze danych._

Procedura sprawdzania metryk z użyciem "k-fold cross-validation" jest zaimplementowana w pakiecie `sklearn` między innymi przez funkcję [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html#sklearn.model_selection.StratifiedKFold).


Funkcja ```cross_val_metrics(df, n_splits=5)``` wypisuje na ekran wartości metryk obliczone z użyciem walidacji krzyżowej. Funkcja przyjmuje na wejściu:
* df - obiekt DataFrame reprezentujący dane
* n_splits - liczba foldów na jaki mają być podzielone nasze dane

Funkcja korzysta ze stworzonych wcześniej funkcji: train_model oraz get_scores

**Proszę:**

* uzupełnić funkcję 'cross_val_metrics' by wypisywała wartości dla metryk precision, recall, accuracy, f1, matthews_corrcoef

In [ ]:
def cross_val_metrics(df, n_splits=5, th=0.5):
    # Wydzielenie danych wejściowych i wyjściowych z df
    X = df[["x","y"]].values
    y = df["label"].values
    metrics_dict_cv = {'PPV': [], 'TPR': [], 'ACC': [], 'F1': [], 'MCC': []}
    # Inicjalizacja StratifiedKFold i pętla po podziałach
...
    
    # Wypisanie średnich i odchyleń standardowych dla każdej miary
    for key in metrics_dict_cv.keys():
        mean_val = np.mean(metrics_dict_cv[key])
        std_val = np.std(metrics_dict_cv[key])
        print(f"{key}: {mean_val:.3f} ± {std_val:.3f}")

cross_val_metrics(df)

Wywołaj funkcję cross_val_metrics dla różnych wielkości zbioru danych treningowych: 5%, 10%, 50%, 70%, 90% całego zbioru danych. Porównaj wyniki.

In [ ]:
...


Wywołaj funkcję cross_val_metrics dla różnych wartości parametru th ```[0.1, 0.5, 0.7]```. Porównaj wyniki.

In [ ]:
...


Teraz proszę stworzyć analogiczną funkcję ```cross_val_roc(df, n_splits=5)```, która zamiast obliczać średnie metryki - stworzy wykres krzywych ROC. Skorzystaj z funkcji ```RocCurveDisplay``` z biblioteki ```sklearn```.

In [ ]:

def cross_val_roc(df, n_splits=5):
    # Wydzielenie danych wejściowych i wyjściowych z df
    X = df[["x","y"]].values
    y = df["label"].values
    fig, ax = plt.subplots(1, 1, figsize=(6,6))
    # Inicjalizacja StratifiedKFold i pętla po podziałach
...
    

cross_val_roc(df)

# Przypadek klas niezrównoważonych.

Do tej pory liczebność obu klas w analizowanych danych była równa. Teraz to zmienimy.

**Proszę:**

* korzystając z początkowych zbiorów `class_0_data` oraz `class_1_data` stworzyć zbiór w którym klasy nie będą równoliczne. Niech klasa "1" będzie 100 razy bardziej liczba niż klasa "0".

**Wskazówka**: 

stwórz nowy obiekt DataFrame z danych dla klas "0" i "1", ale dobierz próbkowanie danych dla poszczególnych klas, tak by w sumarycznym zbiorze klasa "1" była 100 razy bardziej liczba niż klasa "0".

In [34]:
...


**Proszę:**

Korzystając z metod klasy DataFrame (`value_counts()`), sprawdź czy proporcje są zgodne z założeniami.


In [ ]:
...


**Proszę:**

* korzystając z funkcji ```cross_val_metrics(df, n_splits=5)``` oraz ```cross_val_roc(df, n_splits=5)``` wypisz na ekran wartości miar, oraz narysuj krzywe ROC dla danych niezrównoważonych.
* następnie stwórz 'odwrotne' niezbalansowanie klas - niech klasa "0" będzie 100 razy bardziej liczba niż klasa "1" i powtórz punkt wyrzej.
* porównaj metryki dla obu przypadków. Oceń wiarygodność tych metryk dla klas niezbalansowanych.

In [ ]:
print("Przewaga klasy 1:")

...

print("\nPrzewaga klasy 0:")
...
pass

# Stratyfikacja

W przypadku niezbalansowania klas, zwłaszcza dla niezbyt licznych zbiorów, bardzo ważne jest to, aby podział danych na zbiór treningowy i testowy zachował proporcje klas.

Jeśli tego nie dopilnujemy, może się zdarzyć, że zbiór testowy (lub treningowy) nie będzie zawierał żadnych przykładów mniejszościowej klasy, co prowadzi do błędnej oceny jakości modelu.

Funkcje takie jak train_test_split (z parametrem stratify) czy metody walidacji krzyżowej (cross_val_score) dbają o to automatycznie. Warto jednak zobaczyć, jak wygląda różnica między zwykłym losowym wyborem danych a stratyfikacją.

In [ ]:
df_non_equal = pd.concat([df_class_1.sample(frac=0.2), df_class_0.sample(frac=0.01)], ignore_index=True)

print("\nProporcje klas w całym zbiorze:")
print(df_non_equal.label.value_counts(normalize=True))

# --- Podział bez stratyfikacji (losowe 20%) ---
df_test_random = df_non_equal.sample(frac=0.2, random_state=445672)
df_train_random = df_non_equal.drop(df_test_random.index)

print("\nProporcje klas w zbiorze treningowym (losowy podział):")
print(df_train_random["label"].value_counts(normalize=True))
print("\nProporcje klas w zbiorze testowym (losowy podział):")
print(df_test_random["label"].value_counts(normalize=True))

# --- Podział ze stratyfikacją ---
df_train, df_test = train_test_split(
    df_non_equal, 
    test_size=0.2, random_state=42, stratify=df_non_equal["label"]
)

print("\nProporcje klas w zbiorze treningowym (stratyfikacja):")
print(df_train["label"].value_counts(normalize=True))
print("\nProporcje klas w zbiorze testowym (stratyfikacja):")
print(df_test["label"].value_counts(normalize=True))

# Zadanie domowe

Proszę rozważyć dane w których rozkłady klas wyraźnie się różnią, oraz dla takich które się pokrywają: 

* klasy nie rozróżnialne: rozkłady (x,y) dla klas "0" i "1" są takie same:
   
   średnie i wariancje zmiennych $x$ i $y$ dla obu klas są identyczne
  
  
* klasa łatwo separowalne: rozkłady (x,y) dla klas "0" i "1" są bardzo różne: 
   
  średnie zmiennych $x$ i $y$ dla obu klas różnią się o wartość 2, wariancje są takie same i wynoszą 1

W każdym przypadku proszę rozważyć niezbalansowanie klas na dwa sposoby:
* niech klasa "1" będzie 100 razy bardziej liczna niż klasa "0"
* niech klasa "0" będzie 100 razy bardziej liczna niż klasa "1"

**Proszę:**  

* przeprowadzić analizę modelu: wypisać wartości metryk (print_k_cv_scores).
* skomentować wiarygodność wskazań metryk we wszystkich wariantach w komórce **Wnioski** 

***
Przypadek klas nie rozróżnialnych.

In [ ]:
nPoints = 10000    
...
##

***
Przypadek gdy klasy są łatwo odróżnialne:

In [ ]:
nPoints = 10000    
...
##

# Wnioski:

1) Jak się zachowuje model w sytuacji dużej różnicy częstości występowania dwóch klas?

...

2) Jak zachowują się różne metryki w sytuacji dużej różnicy częstości występowania dwóch klas?

...

3) Jak można przeciwdziałać efektowi częstości występowania?

...

4) Czy efekt częstości występowania ma taki sam czy różny wpływ na miary jakości w przypadku, gdy klasy pochodzą z rozkładów, które się znacząco pokrywają lub są mocno rozseparowane?

...
##